In [0]:
!mkdir data

In [3]:
cd data

/content/data


In [4]:
!wget https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/apple.npy
!wget https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/pineapple.npy
!wget https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/grapes.npy
!wget https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/banana.npy

--2018-06-15 14:04:15--  https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/apple.npy
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.141.128, 2607:f8b0:400c:c06::80
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.141.128|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 113462128 (108M) [application/octet-stream]
Saving to: ‘apple.npy’

apple.npy           100%[===================>] 108.21M  87.0MB/s    in 1.2s    

2018-06-15 14:04:16 (87.0 MB/s) - ‘apple.npy’ saved [113462128/113462128]

--2018-06-15 14:04:17--  https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/pineapple.npy
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.141.128, 2607:f8b0:400c:c06::80
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.141.128|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 98055744 (94M) [application/octet-stream]
Saving to: ‘pineapple.np

In [5]:
!ls

apple.npy  banana.npy  grapes.npy  pineapple.npy


In [6]:
cd ..

/content


In [7]:
from sklearn.model_selection import train_test_split as tts
from keras.models import Sequential
from keras.layers import Dense, Dropout, Conv2D, MaxPooling2D, Flatten
from keras.utils import np_utils
from random import randint
import numpy as np
import os
from PIL import Image

Using TensorFlow backend.


In [0]:
# define some constants
N_FRUITS = 4
FRUITS = {0: "Apple", 1: "Banana", 2: "Grape", 3: "Pineapple"}

# number of samples to take in each class
N = 5000

# some other constants
N_EPOCHS = 10

# data files in the same order as defined in FRUITS
files = ["apple.npy", "banana.npy", "grapes.npy", "pineapple.npy"]

In [0]:
def load(dir, reshaped, files):
    "Load .npy or .npz files from disk and return them as numpy arrays. \
    Takes in a list of filenames and returns a list of numpy arrays."

    data = []
    for file in files:
        f = np.load(dir + file)
        if reshaped:
            new_f = []
            for i in range(len(f)):
                x = np.reshape(f[i], (28, 28))
                x = np.expand_dims(x, axis=0)
                x = np.reshape(f[i], (28, 28, 1))
                new_f.append(x)
            f = new_f
        data.append(f)
    return data


def normalize(data):
    "Takes a list or a list of lists and returns its normalized form"

    return np.interp(data, [0, 255], [-1, 1])


def denormalize(data):
    "Takes a list or a list of lists and returns its denormalized form"

    return np.interp(data, [-1, 1], [0, 255])


def visualize(array):
    "Visulaze a 2D array as an Image"
    array = np.reshape(array, (28,28))
    img = Image.fromarray(array)
    return img


def set_limit(arrays, n):
    "Limit elements from each array up to n elements and return a single list"
    new = []
    for array in arrays:
        i = 0
        for item in array:
            if i == n:
                break
            new.append(item)
            i += 1
    return new


def make_labels(N1, N2):
    "make labels from 0 to N1, each repeated N2 times"
    labels = []
    for i in range(N1):
        labels += [i] * N2
    return labels

In [0]:
fruits = load("data/", False, ['apple.npy'])

In [35]:
visualize(fruits[0][0])

In [0]:

#second argument is True for reshaping the image to a 28x28 form. A conv net expects this format.
fruits = load("data/", True, files)
 
#second argument is False because we don't need to reshape the image. An MLP net expects this format.
#fruits = load("data/", False, files)


# limit no of samples in each class to N
fruits = set_limit(fruits, N)

# normalize the values
fruits = map(normalize, fruits)

# define the labels
labels = make_labels(N_FRUITS, N)

# prepare the data
x_train, x_test, y_train, y_test = tts(fruits, labels, test_size=0.05)

# one hot encoding
Y_train = np_utils.to_categorical(y_train, N_FRUITS)
Y_test = np_utils.to_categorical(y_test, N_FRUITS)


In [0]:
model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28,28,1)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(N_FRUITS, activation='softmax'))

In [0]:
# model = Sequential()
# model.add(Dense(units=600, activation='relu', input_dim=784))
# model.add(Dropout(0.3))
# model.add(Dense(units=400, activation='relu'))
# model.add(Dropout(0.3))
# model.add(Dense(units=100, activation='relu'))
# model.add(Dropout(0.3))
# model.add(Dense(units=25, activation='relu'))
# model.add(Dropout(0.3))
# model.add(Dense(units=N_FRUITS, activation='softmax'))

In [14]:
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])


# train
model.fit(np.array(x_train), np.array(Y_train), batch_size=32, epochs=N_EPOCHS)

print "Training complete"

print "Evaluating model"
preds = model.predict(np.array(x_test))

score = 0
for i in range(len(preds)):
    if np.argmax(preds[i]) == y_test[i]:
        score += 1

print "Accuracy: ", ((score + 0.0) / len(preds)) * 100


model.save("fruits"+ ".h5")
print "Model saved"


Epoch 1/10
19000/19000 [==============================] - 9s 496us/step - loss: 0.3054 - acc: 0.8991
Epoch 2/10
19000/19000 [==============================] - 7s 394us/step - loss: 0.1706 - acc: 0.9439
Epoch 3/10
19000/19000 [==============================] - 7s 394us/step - loss: 0.1374 - acc: 0.9557
Epoch 4/10
 7424/19000 [==========>...................] - ETA: 4s - loss: 0.1091 - acc: 0.9620

19000/19000 [==============================] - 7s 389us/step - loss: 0.1116 - acc: 0.9625
Epoch 5/10
19000/19000 [==============================] - 7s 388us/step - loss: 0.0911 - acc: 0.9692
Epoch 6/10
19000/19000 [==============================] - 7s 384us/step - loss: 0.0752 - acc: 0.9752
Epoch 7/10
12800/19000 [===================>..........] - ETA: 2s - loss: 0.0629 - acc: 0.9784

19000/19000 [==============================] - 7s 388us/step - loss: 0.0636 - acc: 0.9777
Epoch 8/10
19000/19000 [==============================] - 8s 400us/step - loss: 0.0537 - acc: 0.9822
Epoch 9/10
19000/19000 [==============================] - 7s 392us/step - loss: 0.0469 - acc: 0.9831
Epoch 10/10
11648/19000 [=================>............] - ETA: 2s - loss: 0.0452 - acc: 0.9838

19000/19000 [==============================] - 7s 394us/step - loss: 0.0443 - acc: 0.9842
Training complete
Evaluating model
Accuracy:  95.7
Model saved
